# Fine-tune Khmer XLS-R on the prepared dataset

This Colab notebook documents the 1,000-row pilot. It prepares the audio, smoke-checks the code, trains the model, and evaluates the untouched test split. The next experiment is a separate 2,000-row Colab notebook.

The notebook saves the materialized WAV cache and training checkpoints in Google Drive so they survive a Colab runtime reset. Full training writes directly to Drive; after an interruption, restore the same files and resume from the latest saved checkpoint.


In [ ]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Upload and organize the ASR files

Upload these individual files from the project Explorer to the Colab Contents root. Run the next cell to create the required folders and move them into place.

- `requirements-asr.txt`
- `scripts/finetune_asr.py`
- `scripts/run_asr_baseline.py`
- `scripts/score_asr_report.py`
- `khmer_transliteration/__init__.py`
- `khmer_transliteration/asr_audio.py`
- `khmer_transliteration/asr_data.py`
- `khmer_transliteration/asr_decoding.py`
- `khmer_transliteration/asr_inference.py`
- `khmer_transliteration/asr_metrics.py`
- `khmer_transliteration/asr_text.py`
- `khmer_transliteration/paths.py`

In [ ]:
%cd /content
from pathlib import Path
import shutil

CONTENT_DIR = Path("/content")
UPLOADS = {
    'requirements-asr.txt': Path('requirements-asr.txt'),
    'finetune_asr.py': Path('scripts/finetune_asr.py'),
    'run_asr_baseline.py': Path('scripts/run_asr_baseline.py'),
    'score_asr_report.py': Path('scripts/score_asr_report.py'),
    'asr_audio.py': Path('khmer_transliteration/asr_audio.py'),
    'asr_data.py': Path('khmer_transliteration/asr_data.py'),
    'asr_decoding.py': Path('khmer_transliteration/asr_decoding.py'),
    'asr_inference.py': Path('khmer_transliteration/asr_inference.py'),
    'asr_metrics.py': Path('khmer_transliteration/asr_metrics.py'),
    'asr_text.py': Path('khmer_transliteration/asr_text.py'),
    'paths.py': Path('khmer_transliteration/paths.py'),
    '__init__.py': Path('khmer_transliteration/__init__.py'),
}

CONTENT_DIR.mkdir(parents=True, exist_ok=True)
for folder in ("scripts", "khmer_transliteration"):
    (CONTENT_DIR / folder).mkdir(parents=True, exist_ok=True)

# Check all uploads before moving any files.
sources = {}
missing = []
for filename, relative_destination in UPLOADS.items():
    destination = CONTENT_DIR / relative_destination
    uploaded_at_root = CONTENT_DIR / filename
    if destination.is_file():
        sources[filename] = destination
    elif uploaded_at_root.is_file():
        sources[filename] = uploaded_at_root
    else:
        missing.append(filename)

if missing:
    missing_list = "\n".join(f"  - {name}" for name in missing)
    raise FileNotFoundError(
        "Upload these individual files to the Colab Contents root, then rerun this cell:\n"
        + missing_list
    )

for filename, relative_destination in UPLOADS.items():
    source = sources[filename]
    destination = CONTENT_DIR / relative_destination
    if source != destination:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(source), str(destination))
    print("Ready:", destination)

print(f"Organized all {len(UPLOADS)} required files under /content.")


In [ ]:
%pip install -q -r /content/requirements-asr.txt


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

drive_data = Path("/content/drive/MyDrive/khmer_asr/voice_1000")
colab_data = Path("/content/data/voice_1000")
colab_data.parent.mkdir(parents=True, exist_ok=True)
if not colab_data.exists():
    shutil.copytree(drive_data, colab_data)

drive_materialized = drive_data / "materialized_training"
colab_materialized = colab_data / "materialized_training"
if drive_materialized.exists() and not colab_materialized.exists():
    shutil.copytree(drive_materialized, colab_materialized)
print("Manifest directory:", colab_data)
print("Files:", sorted(path.name for path in colab_data.iterdir()))
print("Materialized audio available:", colab_materialized.exists())


## 1. Materialize the selected audio

This replays the manifest order on Hugging Face and stores the 793 train, 81 validation, and 126 test WAV files in the Colab runtime. The next cell copies the audio and manifests to Google Drive so a later runtime can restore them. The test files are stored only for evaluation.


In [ ]:
!python /content/scripts/finetune_asr.py --stage materialize --manifest-dir /content/data/voice_1000 --materialized-dir /content/data/voice_1000/materialized_training

In [ ]:
from pathlib import Path
import shutil

local_materialized = Path("/content/data/voice_1000/materialized_training")
drive_materialized = Path("/content/drive/MyDrive/khmer_asr/voice_1000/materialized_training")
drive_materialized.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(local_materialized, drive_materialized, dirs_exist_ok=True)
print("Persisted materialized audio and manifests to:", drive_materialized)


In [ ]:
import json
from pathlib import Path

summary_path = Path("/content/data/voice_1000/materialized_training/materialization_summary.json")
print(json.loads(summary_path.read_text(encoding="utf-8")))

## 2. Smoke training

This runs five optimizer steps on at most 8 training and 4 validation examples. It checks the processor, CTC labels, data collator, GPU, and Trainer before spending compute on the full run.


In [ ]:
from pathlib import Path

required_files = [
    Path("/content/scripts/finetune_asr.py"),
    Path("/content/scripts/run_asr_baseline.py"),
    Path("/content/khmer_transliteration/asr_audio.py"),
    Path("/content/khmer_transliteration/asr_data.py"),
    Path("/content/khmer_transliteration/asr_decoding.py"),
    Path("/content/khmer_transliteration/asr_inference.py"),
    Path("/content/khmer_transliteration/asr_metrics.py"),
    Path("/content/khmer_transliteration/asr_text.py"),
    Path("/content/khmer_transliteration/paths.py"),
]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Upload these files first: {missing}")
print("ASR project files are ready.")

In [ ]:
!python /content/scripts/finetune_asr.py --stage smoke --manifest-dir /content/data/voice_1000 --materialized-dir /content/data/voice_1000/materialized_training --output-dir /content/models/khmer_xlsr_ddd_smoke

In [ ]:
import json
from pathlib import Path

smoke_summary = Path("/content/models/khmer_xlsr_ddd_smoke/smoke_training_summary.json")
print(json.loads(smoke_summary.read_text(encoding="utf-8")))

## 3. Full fine-tuning

Run this only when smoke training succeeds. It uses all train and validation rows for three epochs; the test split remains untouched. The output directory is on Google Drive, so completed checkpoints survive a Colab reset. Checkpoint writes to mounted Drive can take longer.

If Colab disconnects after a checkpoint was saved, reconnect, restore the data and upload the same scripts, then rerun the full-training command with `--resume` added. Keep the same output directory and training settings. The script verifies the saved data and tokenizer metadata before it resumes.

Resume command (use this instead of the first-run command only after a saved checkpoint exists):

```bash
!python /content/scripts/finetune_asr.py --stage full --manifest-dir /content/data/voice_1000 --materialized-dir /content/data/voice_1000/materialized_training --output-dir /content/drive/MyDrive/khmer_asr/models/khmer_xlsr_ddd_1000_clean_resumable --checkpoint-steps 250 --resume
```


In [ ]:
!python /content/scripts/finetune_asr.py --stage full --manifest-dir /content/data/voice_1000 --materialized-dir /content/data/voice_1000/materialized_training --output-dir /content/drive/MyDrive/khmer_asr/models/khmer_xlsr_ddd_1000_clean_resumable --checkpoint-steps 250


In [ ]:
from pathlib import Path

drive_model = Path("/content/drive/MyDrive/khmer_asr/models/khmer_xlsr_ddd_1000_clean_resumable")
summary = drive_model / "full_training_summary.json"
print("Model and checkpoints are stored in:", drive_model)
print("Training summary exists:", summary.is_file())
print("Saved checkpoints:", sorted(path.name for path in drive_model.glob("checkpoint-*")))


## 4. Evaluate the fine-tuned model on the untouched test split

In [ ]:
!python /content/scripts/run_asr_baseline.py --model /content/drive/MyDrive/khmer_asr/models/khmer_xlsr_ddd_1000_clean_resumable --manifest-dir /content/data/voice_1000 --split test --max-samples 0 --output /content/data/voice_1000/finetuned_test_clean.json


In [ ]:
from pathlib import Path
import json
import shutil

report_path = Path("/content/data/voice_1000/finetuned_test_clean.json")
report = json.loads(report_path.read_text(encoding="utf-8"))
print("Model:", report["model"])
print("Metrics:", report["metrics"])

drive_report = Path("/content/drive/MyDrive/khmer_asr/voice_1000/finetuned_test_clean.json")
shutil.copy2(report_path, drive_report)
print("Saved report to:", drive_report)

## 5. Re-score existing reports with Khmer-aware WER

This reads the saved predictions only. It does not reload audio or the ASR model. Review the five tokenization examples in each output report before presenting WER.

In [ ]:
!python /content/scripts/score_asr_report.py --report /content/data/voice_1000/baseline_test.json --output /content/data/voice_1000/baseline_test_rescored.json
!python /content/scripts/score_asr_report.py --report /content/data/voice_1000/finetuned_test_clean.json --output /content/data/voice_1000/finetuned_test_clean_rescored.json

In [ ]:
from pathlib import Path
import shutil

drive_reports = Path("/content/drive/MyDrive/khmer_asr/voice_1000")
drive_reports.mkdir(parents=True, exist_ok=True)
for name in ("baseline_test_rescored.json", "finetuned_test_clean_rescored.json"):
    source = Path("/content/data/voice_1000") / name
    shutil.copy2(source, drive_reports / name)
    print("Saved:", drive_reports / name)

## Next experiment: 2,000 rows in Colab

The 1,000-row pilot is complete. Run `notebooks/finetune_asr_2000.ipynb` to retain its rows and held-out test split, add 1,000 new rows, then fine-tune and evaluate from the same pretrained checkpoint.